In [1]:
import json
import re
import sys
from pathlib import Path

PROJECT_ROOT = Path("../..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import MODELS, PAPILO_PRESOLVED_MODELS
from src.detection.pipstools_backend import PipstoolsDetection

MODEL_SCORING_OUT = Path("/data/energy-system-preprocessing/scoring/models")
MODEL_SCORING_OUT.mkdir(parents=True, exist_ok=True)
PAPILO_SCORING_OUT = Path("/data/energy-system-preprocessing/scoring/papilo")
PAPILO_SCORING_OUT.mkdir(parents=True, exist_ok=True)


def region_k(name: str) -> int | None:
    m = re.match(r"^r(\d+)_", name)
    return int(m.group(1)) if m else None


def detection_params_str(detector: PipstoolsDetection, k: int) -> str:
    vd = detector._var_dense if detector._var_dense is not None else "none"
    return f"k{k}-{detector._hypergraph}-{detector._hg_objective}-vd{vd}"


def score_model(model_dir: Path, out_dir: Path, mps_name: str, k: int) -> bool:
    name = model_dir.name
    mps_in = model_dir / mps_name

    if not mps_in.exists():
        print(f"  SKIP: no {mps_name} in {model_dir}")
        return False

    detector = PipstoolsDetection(mpsreader="highs")
    params = detection_params_str(detector, k)
    out_file = out_dir / f"{name}-{params}.json"


    if out_file.exists():
        print(f"  skipped (already exists)", flush=True)
        return True
    result = detector.detect(mps_in, k=k)
    out_file.write_text(json.dumps(result.to_dict(), indent=2))
    return True

In [2]:
def r_models(root: Path) -> list[tuple[Path, int]]:
    return sorted(
        [(p, region_k(p.name)) for p in root.iterdir() if region_k(p.name) is not None],
        key=lambda x: x[0].name,
    )

orig_models = r_models(MODELS)
papilo_models = r_models(PAPILO_PRESOLVED_MODELS)
print(f"{len(orig_models)} original r-N models, {len(papilo_models)} papilo-presolved r-N models")

440 original r-N models, 440 papilo-presolved r-N models


In [3]:
failed = []

print("=== Original models ===")
for i, (model_dir, k) in enumerate(orig_models, 1):
    print(f"[{i}/{len(orig_models)}] {model_dir.name}  (k={k})", flush=True)
    ok = score_model(model_dir, MODEL_SCORING_OUT, "original.mps", k)
    if not ok:
        failed.append(("original", model_dir.name))
    print(flush=True)

print("=== Papilo-presolved models ===")
for i, (model_dir, k) in enumerate(papilo_models, 1):
    print(f"[{i}/{len(papilo_models)}] {model_dir.name}  (k={k})", flush=True)
    ok = score_model(model_dir, PAPILO_SCORING_OUT, "reduced.mps", k)
    if not ok:
        failed.append(("papilo", model_dir.name))
    print(flush=True)

if failed:
    print(f"\n{len(failed)} model(s) failed: {failed}")
else:
    total = len(orig_models) + len(papilo_models)
    print(f"\nAll {total} models scored successfully.")

=== Original models ===
[1/440] r10_res168_f0.0000_t0.0192  (k=10)
  skipped (already exists)

[2/440] r10_res168_f0.0000_t0.0833  (k=10)
  skipped (already exists)

[3/440] r10_res168_f0.0000_t1.0000  (k=10)
  skipped (already exists)

[4/440] r10_res168_f0.2500_t0.5000  (k=10)
  skipped (already exists)

[5/440] r10_res168_f0.5000_t1.0000  (k=10)
  skipped (already exists)

[6/440] r10_res1_f0.0000_t0.0192  (k=10)
  skipped (already exists)

[7/440] r10_res1_f0.0000_t0.0833  (k=10)
  skipped (already exists)

[8/440] r10_res1_f0.0000_t1.0000  (k=10)
  skipped (already exists)

[9/440] r10_res1_f0.2500_t0.5000  (k=10)
  skipped (already exists)

[10/440] r10_res1_f0.5000_t1.0000  (k=10)
  skipped (already exists)

[11/440] r10_res24_f0.0000_t0.0192  (k=10)
  skipped (already exists)

[12/440] r10_res24_f0.0000_t0.0833  (k=10)
  skipped (already exists)

[13/440] r10_res24_f0.0000_t1.0000  (k=10)
  skipped (already exists)

[14/440] r10_res24_f0.2500_t0.5000  (k=10)
  skipped (already 